In [0]:
# ============================================================
# 04_Data_Quality
# Common Data Quality Validation Notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)
from datetime import datetime


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "f1_catalog"

SILVER_SCHEMA = "silver"
AUDIT_TABLE = "f1_catalog.gold.data_quality_audit"


# ============================================================
# 2. SILVER TABLE CONFIGURATION
# ============================================================

table_config = {

    "drivers": {
        "primary_keys": ["driver_id"]
    },

    "constructors": {
        "primary_keys": ["constructor_id"]
    },

    "circuits": {
        "primary_keys": ["circuit_id"]
    },

    "races": {
        "primary_keys": ["race_id"]
    },

    "results": {
        "primary_keys": ["result_id"]
    },

    "qualifying": {
        "primary_keys": ["qualify_id"]
    },

    "pit_stops": {
        "primary_keys": [
            "race_id",
            "driver_id",
            "stop"
        ]
    },

    "lap_times": {
        "primary_keys": [
            "race_id",
            "driver_id",
            "lap_number"
        ]
    },

    "driver_standings": {
        "primary_keys": [
            "driver_standings_id"
        ]
    },

    "constructor_standings": {
        "primary_keys": [
            "constructor_standings_id"
        ]
    },

    "seasons": {
        "primary_keys": [
            "year"
        ]
    },

    "status": {
        "primary_keys": [
            "status_id"
        ]
    }
}


# ============================================================
# 3. FUNCTION TO INSERT AUDIT RESULT
# ============================================================

def write_audit(
    table_name,
    check_name,
    check_status,
    record_count,
    failed_records,
    message
):

    audit_data = [

        (
            table_name,
            "silver",
            check_name,
            check_status,
            record_count,
            failed_records,
            message,
            datetime.now(),
            None
        )
    ]

    audit_schema = StructType([
        StructField("table_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("check_name", StringType(), True),
        StructField("check_status", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("failed_records", LongType(), True),
        StructField("check_message", StringType(), True),
        StructField("check_timestamp", TimestampType(), True),
        StructField("pipeline_run_id", StringType(), True),
    ])

    audit_df = spark.createDataFrame(
        audit_data,
        audit_schema
    )

    (
        audit_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )


# ============================================================
# 4. FUNCTION - NULL PRIMARY KEY CHECK
# ============================================================

def check_primary_key_nulls(
    df,
    table_name,
    primary_keys,
    record_count
):

    for key in primary_keys:

        if key not in df.columns:

            write_audit(
                table_name,
                f"Primary Key Column Check - {key}",
                "FAIL",
                record_count,
                record_count,
                f"Primary key column {key} does not exist."
            )

            continue


        failed_records = (
            df
            .filter(F.col(key).isNull())
            .count()
        )


        if failed_records == 0:

            status = "PASS"
            message = (
                f"No NULL values found "
                f"in {key}."
            )

        else:

            status = "FAIL"
            message = (
                f"{failed_records} NULL values "
                f"found in {key}."
            )


        write_audit(
            table_name,
            f"Primary Key NULL Check - {key}",
            status,
            record_count,
            failed_records,
            message
        )


# ============================================================
# 5. FUNCTION - DUPLICATE CHECK
# ============================================================

def check_duplicates(
    df,
    table_name,
    primary_keys,
    record_count
):

    existing_keys = [

        key
        for key in primary_keys
        if key in df.columns
    ]


    if not existing_keys:

        write_audit(
            table_name,
            "Duplicate Check",
            "FAIL",
            record_count,
            record_count,
            "Primary key columns not available."
        )

        return


    duplicate_groups = (

        df
        .groupBy(existing_keys)
        .count()
        .filter(
            F.col("count") > 1
        )
    )


    duplicate_records = (

        duplicate_groups
        .agg(
            F.sum(
                F.col("count") - 1
            ).alias("duplicates")
        )
        .collect()[0]["duplicates"]
    )


    duplicate_records = (
        duplicate_records
        if duplicate_records is not None
        else 0
    )


    if duplicate_records == 0:

        status = "PASS"

        message = (
            "No duplicate primary-key records found."
        )

    else:

        status = "FAIL"

        message = (
            f"{duplicate_records} duplicate "
            f"records found."
        )


    write_audit(
        table_name,
        "Duplicate Check",
        status,
        record_count,
        duplicate_records,
        message
    )


# ============================================================
# 6. FUNCTION - EMPTY TABLE CHECK
# ============================================================

def check_empty_table(
    table_name,
    record_count
):

    if record_count > 0:

        status = "PASS"
        failed_records = 0

        message = (
            "Table contains records."
        )

    else:

        status = "FAIL"
        failed_records = 0

        message = (
            "Table contains zero records."
        )


    write_audit(
        table_name,
        "Empty Table Check",
        status,
        record_count,
        failed_records,
        message
    )


# ============================================================
# 7. PROCESS ALL SILVER TABLES
# ============================================================

overall_failed = False


for table_name, config in table_config.items():

    print("\n")
    print("=" * 80)
    print(f"DATA QUALITY CHECK: {table_name}")
    print("=" * 80)


    source_table = (
        f"{CATALOG}."
        f"{SILVER_SCHEMA}."
        f"{table_name}"
    )


    try:

        # ----------------------------------------------------
        # Read Silver
        # ----------------------------------------------------

        df = spark.table(
            source_table
        )


        record_count = df.count()


        print(
            f"Table       : {source_table}"
        )

        print(
            f"Record Count: {record_count}"
        )


        # ----------------------------------------------------
        # 1. Empty Table Check
        # ----------------------------------------------------

        check_empty_table(
            table_name,
            record_count
        )


        # ----------------------------------------------------
        # 2. Primary Key NULL Check
        # ----------------------------------------------------

        check_primary_key_nulls(
            df,
            table_name,
            config["primary_keys"],
            record_count
        )


        # ----------------------------------------------------
        # 3. Duplicate Check
        # ----------------------------------------------------

        check_duplicates(
            df,
            table_name,
            config["primary_keys"],
            record_count
        )


        print(
            f"DQ completed: {table_name}"
        )


    except Exception as e:

        overall_failed = True

        write_audit(
            table_name,
            "Pipeline Execution Check",
            "FAIL",
            0,
            0,
            str(e)
        )


        print(
            f"ERROR processing {table_name}: "
            f"{str(e)}"
        )


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("DATA QUALITY PROCESSING COMPLETED")
print("=" * 80)


# ============================================================
# 9. DISPLAY AUDIT RESULTS
# ============================================================

audit_df = spark.table(
    AUDIT_TABLE
)


display(
    audit_df
    .orderBy(
        F.desc("check_timestamp")
    )
)


# ============================================================
# 10. DISPLAY FAILED CHECKS
# ============================================================

failed_df = (

    audit_df
    .filter(
        F.col("check_status") == "FAIL"
    )
)


print(
    f"Failed checks: {failed_df.count()}"
)


display(
    failed_df
)


print("=" * 80)

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS f1_catalog.gold.data_quality_audit
# (
#     audit_id BIGINT GENERATED ALWAYS AS IDENTITY,

#     table_name STRING,
#     layer STRING,
#     check_name STRING,
#     check_status STRING,

#     record_count BIGINT,
#     failed_records BIGINT,

#     check_message STRING,

#     check_timestamp TIMESTAMP,

#     pipeline_run_id STRING
# )
# USING DELTA;


In [0]:
%sql
SELECT * FROM f1_catalog.gold.data_quality_audit